# FracAtlas fracture detection: ConvNeXt, DINOv2 and YOLO

Companion notebook for [MHKhan50/fracatlas-fracture-benchmark](https://github.com/MHKhan50/fracatlas-fracture-benchmark).
Image-level fracture classification on FracAtlas (4,083 musculoskeletal X-rays, 717 with fractures; CC BY 4.0) with a small CNN,
ConvNeXt-Tiny, DINOv2-S (frozen probe and fine-tuned), three seeds each, plus YOLO localisation, calibration and abstention.

**Research only. Not a medical device.**

Kaggle settings: **GPU T4**, **Internet on**. *Save Version → Save & Run All*. A smoke test runs first; the full run takes about 3 hours.
If figshare is unreachable, attach a Kaggle copy of FracAtlas (Add Input) and it is used instead.
Outputs: `/kaggle/working/reports/` and `/kaggle/working/fracatlas_reports.zip`.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/MHKhan50/fracatlas-fracture-benchmark.git"
os.chdir("/kaggle/working")
if not os.path.exists("fracatlas-fracture-benchmark"):
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
os.chdir("fracatlas-fracture-benchmark")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
import torch, timm, ultralytics
print("torch", torch.__version__, "| timm", timm.__version__, "| ultralytics", ultralytics.__version__, "| cuda", torch.cuda.is_available())
DATA, RUNS, REPORTS = "/kaggle/working/data", "/kaggle/working/runs", "/kaggle/working/reports"
def fa(*args):
    """Run one CLI step, stream its output into the notebook and stop with the error text if it fails."""
    proc = subprocess.Popen([sys.executable, "-m", "fracatlas_bench.cli", *args, "--data-dir", DATA],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in proc.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    if proc.wait():
        raise RuntimeError("step failed: " + " ".join(args) + "\n" + "".join(lines[-30:]))

## 1. Data (downloaded from figshare, not redistributed) and the fixed split

In [ ]:
fa("download")
fa("split", "--out", REPORTS)

## 2. Smoke test: every step on a small subset, one epoch

In [ ]:
S = "/kaggle/working/smoke"
fa("probe", "--limit", "120", "--out", S)
for m in ("cnn", "convnext_tiny", "dinov2_small"):
    fa("train", "--model", m, "--limit", "120", "--epochs", "1", "--seeds", "0", "--out", S)
fa("detect", "--limit", "60", "--epochs", "1", "--out", S)
fa("analyse", "--limit", "120", "--out", S, "--reports", S + "/reports")
import shutil; shutil.rmtree(S, ignore_errors=True)

## 3. Full run: frozen probe, three classifiers x three seeds, YOLO

In [ ]:
# Reuse finished steps from an earlier version of this notebook, if its output is attached (Add Input).
import glob, os, shutil
os.makedirs(RUNS, exist_ok=True)
for d in sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/*_meta.json", recursive=True)}):
    for f in glob.glob(d + "/*_logits.npz") + glob.glob(d + "/*_meta.json") + glob.glob(d + "/*_best.pt"):
        if not os.path.exists(os.path.join(RUNS, os.path.basename(f))):
            shutil.copy(f, RUNS); print("restored", os.path.basename(f))
for w in glob.glob("/kaggle/input/**/yolo_runs/train/weights/best.pt", recursive=True)[:1]:
    os.makedirs(RUNS + "/yolo_runs/train/weights", exist_ok=True)
    shutil.copy(w, RUNS + "/yolo_runs/train/weights/best.pt"); print("restored YOLO weights")

fa("probe", "--out", RUNS)
for m in ("cnn", "convnext_tiny", "dinov2_small"):
    fa("train", "--model", m, "--seeds", "0,1,2", "--out", RUNS)
fa("detect", "--epochs", "60", "--out", RUNS)

## 4. Analysis, released weights and zip

In [ ]:
fa("analyse", "--out", RUNS, "--reports", REPORTS)
import glob, shutil, json
os.makedirs(REPORTS + "/runs", exist_ok=True)
for f in glob.glob(RUNS + "/*_logits.npz") + glob.glob(RUNS + "/*_meta.json") + glob.glob(RUNS + "/yolo_*.json") + glob.glob(RUNS + "/yolo_test_scores.npz"):
    shutil.copy(f, REPORTS + "/runs/")
for p in glob.glob(DATA + "/cache_*.npy"):
    os.remove(p)
shutil.make_archive("/kaggle/working/fracatlas_reports", "zip", REPORTS)
print(open(REPORTS + "/analysis.md").read())
print(json.dumps(json.load(open(REPORTS + "/analysis.json"))["released_weights"], indent=1))